In [1]:
import sys
import os

# Import the function we wrote
from scripts.prepare_data import prepare_data

# Execute the reverse geocoding
prepare_data()

Loading coordinates from c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\ground_truth_coordinates.csv...
Creating GeoDataFrame...
Loading country boundaries from c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\country_boundaries.geojson...
Performing spatial join (this may take a minute)...
Found 2877 points outside strict boundaries (likely coasts/islands). Finding nearest country...
Final missing countries: 0
Saving enriched data to c:\Users\Yash T\Desktop\geoguessr\geolocation-prediction\training_dataset\noised_dataset\enriched_training_data.csv...
Done! Data preparation is complete.


In [2]:
import pandas as pd

# Load the enriched dataset
df = pd.read_csv('training_dataset/noised_dataset/enriched_training_data.csv')

# Group by country name, count the occurrences, and sort them
country_counts = df['country_name'].value_counts()

# Display the top 10 countries with the highest number of images
print("Top 10 countries with the most images:")
print(country_counts.head(10))

Top 10 countries with the most images:
country_name
United States of America    2135
Russia                      1916
Brazil                      1599
Australia                   1457
Canada                      1187
Argentina                   1015
South Africa                 684
India                        380
Chile                        378
France                       312
Name: count, dtype: int64


In [4]:
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from scripts.datasets import GeoguessrDataset

# Define basic image transformations
# Resizing to 224x224 and converting to PyTorch Tensor format
transform = A.Compose([
    A.Resize(224, 224),
    ToTensorV2()
])

# Define paths
base_dir = os.path.abspath('.')
csv_path = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'enriched_training_data.csv')
image_dir = os.path.join(base_dir, 'training_dataset', 'noised_dataset', 'images')

# Initialize the Dataset
dataset = GeoguessrDataset(csv_path=csv_path, image_dir=image_dir, transform=transform)

print(f"Dataset successfully loaded with {len(dataset)} images!")
print(f"Found {dataset.get_num_classes()} unique countries.")

# Initialize the DataLoader (to fetch images in batches of 4)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Fetch one batch of data
batch = next(iter(dataloader))

print("\n--- Batch Information ---")
print("Image Tensor Shape:", batch['image'].shape) # Should be [4, 3, 224, 224]
print("Country Labels:", batch['country_label'])
print("Latitudes:", batch['latitude'])
print("Longitudes:", batch['longitude'])

Dataset successfully loaded with 19002 images!
Found 191 unique countries.

--- Batch Information ---
Image Tensor Shape: torch.Size([4, 3, 224, 224])
Country Labels: tensor([  1, 154,  22,   7])
Latitudes: tensor([ 59.6411, -31.0765, -22.1602, -24.2848])
Longitudes: tensor([ 20.1841,  26.8514, -48.4643, -65.9005])
